In [1]:
from openai import OpenAI
import os
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)


In [2]:
db = {} 

In [3]:
import tqdm, os
import random
import time
reviews = os.listdir("../human_reviews_2026")
def build_embedding(i): 
    while True:
        try:
            if reviews[i] in db.keys():
                print(f"Embedding for {reviews[i]} already exists, skipping...")
                return db[reviews[i]] 
            else:
                with open(f"../human_reviews_2026/{reviews[i]}", "r") as f:
                    content = f.read()

                embedding = client.embeddings.create(
                    model="google/gemini-embedding-001",
                    input=content,
                    encoding_format="float"
                )
                db[reviews[i]] = embedding.data[0].embedding
            return embedding.data[0].embedding
        except Exception as e:
            print(f"Error embedding {reviews[i]}: {e}. Retrying...")
            time.sleep(random.uniform(1, 10))


In [4]:
db.keys()

dict_keys([])

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(build_embedding, i): i for i in range(len(reviews))}
    results = {}
    for f in tqdm.tqdm(as_completed(futures), total=len(futures)):
        idx = futures[f]
        results[idx] = f.result() 

  0%|          | 1/18590 [00:01<6:06:37,  1.18s/it]

Error embedding 0yOsSMU1eY.md: 'NoneType' object is not subscriptable. Retrying...


 33%|███▎      | 6189/18590 [03:14<06:07, 33.75it/s]

Error embedding 58681hX7oH.md: 'NoneType' object is not subscriptable. Retrying...
Error embedding R2YwWxO2U0.md: 'NoneType' object is not subscriptable. Retrying...


100%|██████████| 18590/18590 [09:38<00:00, 32.15it/s]


In [6]:
keys = list(db.keys()) 
values = list(db.values())

In [7]:
import numpy as np

values = np.array(values)

In [13]:
query_embedding = client.embeddings.create( 
    model="google/gemini-embedding-001",
    input="very low score paper",
    encoding_format="float"  
)

In [18]:
keys[(np.array(query_embedding.data[0].embedding) @ values.T).argmax()]

'w73feIekdO.md'

In [ ]:
filename = keys[(np.array(query_embedding.data[0].embedding) @ values.T).argsort()[-1]]
with open(f"../human_reviews/{filename}", "r") as f:
    print(f"\nMost relevant review for query:\n{f.read()}")

In [8]:
with open(f"../new/human_reviews_embeddings_2026.pkl", "wb") as f:
    import pickle
    pickle.dump(db, f)

In [23]:
!cp ./human_reviews_embeddings.pkl ../new/ 